In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(df.shape)
df.to_csv("breast_cancer.csv", index=False)

(569, 31)


In [2]:
import pandas as pd
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# ---------- STEP 1: LOAD DATA ----------
df = pd.read_csv("breast_cancer.csv")

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create model folder if not exists
os.makedirs("model", exist_ok=True)

# ---------- STEP 2: DEFINE MODELS ----------
models = {
    "Logistic_Regression": LogisticRegression(max_iter=5000),
    "Decision_Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive_Bayes": GaussianNB(),
    "Random_Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    # XGBoost alternative that works everywhere:
    "XGBoost": GradientBoostingClassifier(random_state=42),
}

# ---------- STEP 3: TRAIN + EVALUATE ----------
results = []

for name, model in models.items():
    print(f"\nTraining: {name}")

    # Train
    model.fit(X_train, y_train)

    # Predict
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    # Metrics
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "AUC": roc_auc_score(y_test, probs),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
        "MCC": matthews_corrcoef(y_test, preds),
    }

    results.append(metrics)

    # Save model
    joblib.dump(model, f"model/{name}.pkl")

    # Print nicely
    for k, v in metrics.items():
        if k != "Model":
            print(f"{k}: {v:.4f}")

# ---------- STEP 4: SAVE RESULTS TABLE ----------
results_df = pd.DataFrame(results)
results_df.to_csv("model_results.csv", index=False)

print("\n===== FINAL RESULTS =====")
print(results_df)

print("\nAll models saved in the 'model/' folder.")
print("Metrics saved in model_results.csv")


Training: Logistic_Regression
Accuracy: 0.9561
AUC: 0.9977
Precision: 0.9459
Recall: 0.9859
F1: 0.9655
MCC: 0.9068

Training: Decision_Tree
Accuracy: 0.9474
AUC: 0.9440
Precision: 0.9577
Recall: 0.9577
F1: 0.9577
MCC: 0.8880

Training: KNN
Accuracy: 0.9561
AUC: 0.9959
Precision: 0.9342
Recall: 1.0000
F1: 0.9660
MCC: 0.9086

Training: Naive_Bayes
Accuracy: 0.9737
AUC: 0.9984
Precision: 0.9595
Recall: 1.0000
F1: 0.9793
MCC: 0.9447

Training: Random_Forest
Accuracy: 0.9649
AUC: 0.9953
Precision: 0.9589
Recall: 0.9859
F1: 0.9722
MCC: 0.9253

Training: XGBoost
Accuracy: 0.9561
AUC: 0.9951
Precision: 0.9583
Recall: 0.9718
F1: 0.9650
MCC: 0.9064

===== FINAL RESULTS =====
                 Model  Accuracy       AUC  Precision    Recall        F1  \
0  Logistic_Regression  0.956140  0.997707   0.945946  0.985915  0.965517   
1        Decision_Tree  0.947368  0.943990   0.957746  0.957746  0.957746   
2                  KNN  0.956140  0.995906   0.934211  1.000000  0.965986   
3          Naive_